In [1]:
from _src.Utils.CompositionLoader import CompositionExcelLoader
from _src.Composition.CompositionV2 import Composition
from _src.EOS.BaseEOS import EOSType

## Загрузка состава из БД

In [2]:
composition_db = Composition.from_db(r'C:\Users\user\Desktop\PVTcalc\models.json')

In [3]:
comp_KRSNL = composition_db.KRSNL_PVTSIM

In [4]:
comp_KRSNL.normalize_composition()

In [5]:
list(comp_KRSNL.composition_data['b'].values())

[2.4036014744301757,
 2.6647238394146395,
 2.678640457356527,
 4.042696857798534,
 5.630209200955264,
 7.2318678657829105,
 7.239220484017923,
 8.782207061403382,
 9.008205779814855,
 10.569725128221672,
 11.12752056772811,
 12.505473975552476,
 14.130823038987433,
 15.883554544951771,
 17.66685442808586,
 19.53985072003438,
 21.250824822345646,
 23.09329999635536,
 25.170073818659002,
 27.213679915152603,
 29.286308002554648,
 31.057862587840575,
 32.72540714548372,
 34.656199714480834,
 36.67622635664682,
 38.57287705362734,
 40.54526820865336,
 42.50775576964399,
 44.704421313632025,
 46.8345446184484,
 49.04874740487831,
 51.25521910259816,
 53.2570782831901,
 55.02541007934495,
 56.94152869144317,
 58.81073658595427,
 60.73180764241405,
 62.604212623866054,
 64.63210113675625,
 66.61206437013463]

In [6]:
comp_PRRZLM = composition_db.PRRZLM_MDT_TEST


## Старая загрузка составов

In [ ]:
xl_loader = CompositionExcelLoader(r'C:\Users\user\Desktop\PVT_TSU\diss\krsnln.xlsx')
comp_dict = xl_loader.load(header=True, sheet = 'to_model')

In [ ]:
composition = Composition(comp_dict, eos_name= EOSType.PREOS)
composition.evaluate_composition_data({'critical_temperature': 'kesler lee',
                                                    'critical_pressure': 'kesler lee',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})

composition.edit_component_properties('C36', 'molar_mass', 650)

In [ ]:
composition2 = Composition(comp_dict)
composition2.evaluate_composition_data({'critical_temperature': 'pedersen',
                                                    'critical_pressure': 'pedersen',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})

In [ ]:
composition2.normalize_composition()

In [ ]:
composition.composition_data.keys()

In [ ]:
for i in composition.composition_data['acentric_factor'].values():
    print(i)

## Расчет флеша

In [ ]:
from _src.VLE.Flash import Flash
from _src.Utils.Conditions import Conditions

In [ ]:
conds_stable = Conditions(250, 390)

In [ ]:
comp_KRSNL.T = conds_stable.t

In [ ]:
flash_obj1 = Flash(comp_KRSNL, conds_stable)

In [ ]:
flash_res1 = flash_obj1.calculate()

In [ ]:
flash_res1

In [ ]:
from _src.PhaseStability.PhaseIdentificator import PhaseIndentificator

In [ ]:
phase_iden = PhaseIndentificator(flash_obj1)

In [ ]:
phase_iden.identify_phase()

In [ ]:
from _src.Experiments.DLE2 import DLE

In [ ]:
dle_obj = DLE(comp_KRSNL, [330, 300, 270, 240, 210, 150, 100, 50, 30, 10, 5], 300, 110)

In [ ]:
dle_data = dle_obj.calculate()

In [ ]:
dle_data

In [ ]:
dle_obj._dle_df

In [ ]:
from _src.Experiments.SeparatorTest2 import SeparatorTest

In [ ]:
sep_test = SeparatorTest(comp_KRSNL, [210, 5.5, 1.6], [110, 25, 25], 300, 110)

In [ ]:
sep_test.calculate()

In [ ]:
sep_test._dle_df

## Работа через интерфейс  Model

In [ ]:
from _src.CompositionalModel.CompositionalModelV2 import CompositionalModel
from _src.Utils.Conditions import Conditions

In [ ]:
model = CompositionalModel(comp_KRSNL)

In [ ]:
conds = Conditions(100, 383)

In [ ]:
model.flash(110,373)

In [ ]:
model.flash(111, 373)

In [ ]:
model.result_store_object._results[0]

In [ ]:
model.result_store_object.get_by_module('Flash')

In [ ]:
from _src.Utils.Export2 import ModelJSONDB

In [ ]:
model_jsn = ModelJSONDB()

In [ ]:
model_jsn.export('PRRZLM_MDT_TEST', 'BASE', composition.composition, composition.composition_data,
                 composition.eos_name, None, field='PRRZLM')

In [ ]:
model_jsn.save()

## Новый переписанный модуль по расчету Psat

In [ ]:
from _src.PhaseDiagram.new_methodv2 import SaturationPressure

In [ ]:
sat_p = SaturationPressure(composition, t_K= 110+273.15, p_max_bar=600)

In [ ]:
sat_p.sp_convergence_loop()

In [ ]:
sat_p.p_i

In [ ]:
import numpy as np

t_arr = np.linspace(273.15, 550+273.15, 20)

for t in t_arr:
    sat_p = SaturationPressure(composition=composition,
                               t_K=t, p_max_bar= 500)
    res_sp = sat_p.sp_convergence_loop()
    res_dp = sat_p.dp_convergence_loop()

    print('====')
    print(f'RESULT: PSAT:{res_sp}, PDEW:{res_dp}, T:{t-273.15}')
    print('====')


## Давление насыщения, давление конденсации и фазовая

In [ ]:
from _src.PhaseDiagram.BubblePointPressure import BubblePointCalculator

from _src.PhaseDiagram.DewPressure import DewPointCalculator
from _src.PhaseDiagram.PhaseEnvelope import PhaseDiagramCalculator
from _src.PhaseDiagram.CriticalPoint import CriticalPointCalculator
from _src.PhaseDiagram.PhaseEnvelopeFromStability import PhaseEnvelopeFromStability

### Psat

In [ ]:
p_sat_calc = BubblePointCalculator(comp_KRSNL, 110+273.15)
p_sat_calc.calculate()

### Pdew

In [ ]:
p_dew_calc = DewPointCalculator(comp_KRSNL, 330 + 273.15, P_guess= 260, dew_point_type = '')
p_dew_calc.calculate()

### EnvelopeFromStability

In [ ]:
comp_KRSNL.eos_name = EOSType.BRSEOS

In [ ]:
comp_KRSNL.normalize_composition()

In [ ]:
comp_PRRZLM.eos_name = EOSType.BRSEOS

In [ ]:
phase_env_from_stab = PhaseEnvelopeFromStability(comp_KRSNL, max_pressure=600, max_temperature=550, pressure_points=30, temperature_points=30)

In [ ]:
phase_env_from_stab.run_parallel()

In [ ]:
phase_env_from_stab.plot()

### Envelope

In [ ]:
phase_envelope = PhaseDiagramCalculator(comp_KRSNL, 273.15, 550 + 273.15)
phase_envelope.calculate_phase_envelope()

In [ ]:
phase_envelope.plot_phase_diagram()

#### From VBA

In [ ]:
import numpy as np
import logging
from typing import Dict, Optional
from _src.Composition.CompositionV2 import Composition
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest

logger = logging.getLogger(__name__)


class PhaseDiagramCalculator:
    """
    Расчет фазовой диаграммы методом бисекции (как в VBA).
    """
    
    def __init__(self, composition: Composition, T_min: float, T_max: float, dt: float = 5.0):
        self.composition = composition
        self.T_min = T_min
        self.T_max = T_max
        self.dt = dt
        
        self.bubble_points = {'T': [], 'P': []}
        self.dew_points = {'T': [], 'P': []}
    
    def _stability_test(self, P: float, T: float) -> Dict:
        """
        Тест стабильности Michelsen (как в VBA).
        Возвращает: stable, Sv, Sl, K_v, K_l
        """
        stab = TwoPhaseStabilityTest(self.composition, P, T)
        stab.calculate_phase_stability()
        
        return {
            'stable': stab.stable,
            'S_v': stab.S_v,
            'S_l': stab.S_l,
            'K_v': stab.k_values_vapour,
            'K_l': stab.k_values_liquid,
            'K_flash': stab.k_values_for_flash
        }
    
    def _find_saturation_pressure(self, T: float, P_prev: float) -> Optional[Dict]:
        """
        Поиск давления насыщения методом бисекции (как в VBA).
        """
        self.composition.T = T
        
        # Начальные границы
        P_max = 2.0 * P_prev if P_prev > 0 else 300.0
        P_min = 0.0
        P = P_prev if P_prev > 0 else 100.0
        
        # Тест стабильности при текущем P
        stab = self._stability_test(P, T)
        
        if stab['stable']:
            # Система стабильна — нужно найти давление, где она становится нестабильной
            # Идем вниз (bubble) или вверх (dew)
            logger.info(f"T={T-273.15:.1f}°C: Stable at P={P:.2f} bar, searching...")
            
            # Пробуем найти нестабильную область
            for _ in range(20):
                P_test = P * 0.5
                stab_test = self._stability_test(P_test, T)
                
                if not stab_test['stable']:
                    P_max = P
                    P_min = P_test
                    break
                
                P = P_test
            else:
                return None  # Не нашли нестабильную область
        
        # Бисекция
        for _ in range(50):
            if P_max - P_min < 1e-6:
                break
            
            P = (P_max + P_min) / 2.0
            stab = self._stability_test(P, T)
            
            if stab['stable']:
                P_max = P
            else:
                P_min = P
        
        # Определяем тип точки насыщения
        if stab['S_v'] > 1.0 and stab['S_v'] > stab['S_l']:
            # Bubble point
            return {'type': 'bubble', 'P': P, 'S_v': stab['S_v'], 'S_l': stab['S_l']}
        elif stab['S_l'] > 1.0 and stab['S_l'] > stab['S_v']:
            # Dew point
            return {'type': 'dew', 'P': P, 'S_v': stab['S_v'], 'S_l': stab['S_l']}
        else:
            return None
    
    def calculate(self) -> Dict:
        """
        Основной расчет фазовой диаграммы.
        """
        logger.info(f"Starting phase diagram calculation: T=[{self.T_min-273.15:.1f}, {self.T_max-273.15:.1f}]°C")
        
        P_prev = 100.0  # Начальное давление
        
        T = self.T_max
        while T >= self.T_min:
            logger.info(f"\n{'='*60}")
            logger.info(f"T = {T-273.15:.2f}°C")
            
            result = self._find_saturation_pressure(T, P_prev)
            
            if result is not None:
                if result['type'] == 'bubble':
                    self.bubble_points['T'].append(T)
                    self.bubble_points['P'].append(result['P'])
                    logger.info(f"  Bubble point: P = {result['P']:.4f} bar")
                else:
                    self.dew_points['T'].append(T)
                    self.dew_points['P'].append(result['P'])
                    logger.info(f"  Dew point: P = {result['P']:.4f} bar")
                
                P_prev = result['P']
            else:
                logger.info(f"  No saturation point found")
            
            T -= self.dt
        
        return {
            'bubble': self.bubble_points,
            'dew': self.dew_points
        }



In [ ]:
phase_diag_vba = PhaseDiagramCalculator(comp_KRSNL, T_min=280, T_max=500 + 273.15, dt=20)

In [ ]:
phase_diag_vba.calculate()

In [ ]:
import pandas as pd
upp_df = pd.DataFrame.from_dict(phase_diag_vba.bubble_points)
low_df = pd.DataFrame.from_dict(phase_diag_vba.dew_points)

In [ ]:
upp_df

In [ ]:
low_df

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
full_df = pd.concat([upp_df, low_df], ignore_index=True)


In [ ]:
plt.scatter(full_df['T'], full_df['P'])

### CritPoint

In [ ]:
crit_p = CriticalPointCalculator(comp_KRSNL)
crit_p.calculate()

#### Parallel

In [ ]:
crit_p_par = CriticalPointCalculatorParallel(comp_KRSNL)
crit_p_par.calculate()

### Версия Дениса по расчету Pdew - нужно передавать K-факторы отдельно (где я их должен взять :/)

In [ ]:
from _src.PhaseDiagram.SaturationPressure_den_v import DewPointCalculator

In [ ]:
sat_p_den = DewPointCalculator(comp_KRSNL, T=350 + 273.15)

In [ ]:
sat_p_den.calculate()

## Critical Point

In [ ]:
# _src/PVT/CriticalPointCalculatorV1.py
import numpy as np
from _src.Composition.CompositionV2 import Composition
from _src.EOS.BrusilovskiyEOSV2 import BrusilovskiyEOS
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest
from _src.Utils.Errors import StopIterationError

class CriticalPointCalculator:
    """
    Расчет критической точки многокомпонентной смеси методом сканирования (T, P) сетки.
    Использует ваш TwoPhaseStabilityTest для определения перехода двухфазной области в однофазную.
    
    Критерий критической точки:
      - S_v -> 1.0 и S_l -> 1.0 (фазовые фракции стремятся к единице)
      - K-values сходятся к тривиальному решению (K_i -> 1.0)
    """
    def __init__(
        self, 
        composition: Composition, 
        t_range: tuple = None, 
        p_range: tuple = None, 
        t_step: float = 5.0, 
        p_step: float = 1.0, 
        tol_metric: float = 1e-6
    ):
        self.comp = composition
        self.t_step = t_step
        self.p_step = p_step
        self.tol_metric = tol_metric

        # Автоматический расчет диапазонов, если не заданы
        tc_vals = np.array([composition.composition_data['critical_temperature'][c] for c in composition.composition])
        pc_vals = np.array([composition.composition_data['critical_pressure'][c] for c in composition.composition])

        self.t_min, self.t_max = t_range or (tc_vals.min() * 0.8, tc_vals.max() * 1.15)
        self.p_min, self.p_max = p_range or (1.0, pc_vals.max() * 1.3)

        self._best_T = None
        self._best_P = None
        self._best_stab = None
        self._min_metric = float('inf')
        self._history = []

    def calculate(self) -> dict:
        """
        Запускает поиск по сетке (T, P).
        Возвращает словарь с результатами или None, если поиск не дал сходимости.
        """
        t_grid = np.arange(self.t_min, self.t_max, self.t_step)
        p_grid = np.arange(self.p_min, self.p_max, self.p_step)

        for T in t_grid:
            for P in p_grid:
                try:
                    self.comp.T = T
                    stab = TwoPhaseStabilityTest(self.comp, P, T)
                    stab.calculate_phase_stability()

                    # Метрика близости к критической точке
                    s_v = stab.S_v if stab.S_v is not None else 100.0
                    s_l = stab.S_l if stab.S_l is not None else 100.0
                    
                    # Отклонение S от 1.0
                    metric = abs(s_v - 1.0) + abs(s_l - 1.0)

                    # Бонус за сходимость в тривиальное решение (K -> 1)
                    if stab.convergence_trivial_solution_v and stab.convergence_trivial_solution_l:
                        metric *= 0.01

                    if metric < self._min_metric:
                        self._min_metric = metric
                        self._best_T, self._best_P, self._best_stab = T, P, stab

                        # Ранний выход, если достигнута высокая точность
                        if self._min_metric < self.tol_metric:
                            break

                except (StopIterationError, ZeroDivisionError, ValueError, RuntimeError):
                    # Игнорируем точки, где EOS или тест стабильности не сходятся
                    continue
            if self._min_metric < self.tol_metric:
                break

        if self._best_stab is None:
            return None

        return {
            'T_crit': float(self._best_T),
            'P_crit': float(self._best_P),
            'S_v': float(self._best_stab.S_v),
            'S_l': float(self._best_stab.S_l),
            'K_v': self._best_stab.k_values_vapour,
            'K_l': self._best_stab.k_values_liquid,
            'metric': float(self._min_metric),
            'is_trivial_converged': bool(
                self._best_stab.convergence_trivial_solution_v and 
                self._best_stab.convergence_trivial_solution_l
            )
        }

    def refine_with_newton(self, T0: float, P0: float, max_iter: int = 20):
        """
        ЗАГОТОВКА НА БУДУЩЕЕ: уточнение точки методом Ньютона/оптимизации.
        После тестов сетки здесь можно подключить scipy.optimize.root или 
        аналитический расчет производных из BrusilovskiyEOSV2.
        """
        pass  # Реализуем на следующем этапе по вашему запросу

In [ ]:
crit_point_obj = CriticalPointCalculator(composition)

In [ ]:
crit_point_obj.calculate()

### Через параллельный счет

In [ ]:
import numpy as np
from typing import Optional, Dict, Any, Tuple
from joblib import Parallel, delayed
from _src.Composition.CompositionV2 import Composition
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest


def _evaluate_grid_point(composition: Composition, t: float, p: float) -> Optional[Dict[str, Any]]:
    """
    Вычисляет метрику близости к критической точке для заданных (T, P).
    Вынесена на уровень модуля для корректной сериализации в joblib.
    """
    try:
        comp = composition.new_composition(composition.composition, deep_copy=True)
        comp.T = t
        stab = TwoPhaseStabilityTest(comp, p, t)
        stab.calculate_phase_stability()

        s_v = stab.S_v
        s_l = stab.S_l

        if s_v is None or s_l is None:
            return None

        # 1. Базовая метрика: суммарное отклонение фазовых фракций от 1.0
        metric = abs(s_v - 1.0) + abs(s_l - 1.0)

        # 2. Бонус за сходимость в тривиальное решение (K_i -> 1)
        if stab.convergence_trivial_solution_v and stab.convergence_trivial_solution_l:
            metric *= 0.01

        # 3. Штраф за отсутствие сходимости хотя бы одного цикла (v/l)
        conv_v = stab.convergence_v or stab.convergence_trivial_solution_v
        conv_l = stab.convergence_l or stab.convergence_trivial_solution_l
        if not (conv_v and conv_l):
            metric += 10.0

        return {
            'T': float(t) - 273.15,
            'P': float(p),
            'metric': float(metric),
            'S_v': float(s_v),
            'S_l': float(s_l),
            'K_v': stab.k_values_vapour,
            'K_l': stab.k_values_liquid,
            'is_trivial': bool(stab.convergence_trivial_solution_v and stab.convergence_trivial_solution_l),
            'stable': stab.stable
        }
    except Exception:
        # Игнорируем точки, где EOS не сходится, возникает деление на ноль или StopIterationError
        return None




class CriticalPointCalculatorParrallel:
    """
    Расчет критической точки многокомпонентной смеси методом сканирования сетки (T, P).
    Использует joblib для параллельного выполнения независимых расчетов стабильности.
    Автоматически определяет границы поиска по Tc и Pc компонентов, если они не заданы явно.
    """
    def __init__(
        self,
        composition: Composition,
        t_range: Optional[Tuple[float, float]] = None,
        p_range: Optional[Tuple[float, float]] = None,
        t_step: float = 5.0,
        p_step: float = 5.0,
        n_jobs: int = -1,
        backend: str = 'loky'
    ):
        self.composition = composition
        self.t_step = t_step
        self.p_step = p_step
        self.n_jobs = n_jobs
        self.backend = backend

        comp_data = composition.composition_data
        components = tuple(composition.composition.keys())

        # Проверка наличия критических параметров
        if not comp_data.get('critical_temperature') or not comp_data.get('critical_pressure'):
            raise RuntimeError(
                "Необходимо предварительно вызвать composition.evaluate_composition_data(), "
                "чтобы заполнить critical_temperature и critical_pressure."
            )

        # Автоматическая генерация границ
        tc_vals = np.fromiter(
            (comp_data['critical_temperature'][c] for c in components), dtype=np.float64
        )
        pc_vals = np.fromiter(
            (comp_data['critical_pressure'][c] for c in components), dtype=np.float64
        )

        self.t_min, self.t_max = t_range or (float(tc_vals.min() * 0.5), float(tc_vals.max() * 1.5))
        self.p_min, self.p_max = p_range or (1.0, float(pc_vals.max() * 3))

    def calculate(self) -> Optional[Dict[str, Any]]:
        """
        Запускает параллельный поиск по сетке и возвращает параметры точки 
        с минимальной метрикой близости к критическому состоянию.
        """
        # Генерация сетки (с учётом погрешности float для включения верхней границы)
        t_grid = np.arange(self.t_min, self.t_max + self.t_step * 0.5, self.t_step)
        p_grid = np.arange(self.p_min, self.p_max + self.p_step * 0.5, self.p_step)
        points = [(t, p) for t in t_grid for p in p_grid]

        # Параллельное выполнение
        with Parallel(n_jobs=self.n_jobs, backend=self.backend) as parallel:
            results = parallel(
                delayed(_evaluate_grid_point)(self.composition, t, p)
                for t, p in points
            )

        # Фильтрация успешных расчетов
        valid_results = [r for r in results if r is not None]
        if not valid_results:
            return None

        # Выбор точки с наименьшей метрикой
        return min(valid_results, key=lambda x: x['metric'])


In [ ]:
crit_point_obj_par = CriticalPointCalculatorParrallel(comp_KRSNL)
crit_point_obj_par.calculate()

# CCE в параллель - неэффективно

In [7]:
from _src.Experiments.CCE2 import CCE

In [10]:
cce = CCE(comp_KRSNL, [250, 200], 110)

In [11]:
cce.calculate()

2026-06-18 16:45:16,186 | _src.PhaseDiagram.new_methodv2 | INFO | Инициализация: T=383.15 K, P_min=0.0100, P_max=1000.0000
2026-06-18 16:45:16,187 | _src.PhaseDiagram.new_methodv2 | INFO | Запуск поиска давления насыщения
2026-06-18 16:45:16,188 | _src.PhaseDiagram.new_methodv2 | INFO | 🔍 Поиск bracket: start=[0.010, 1000.000]
2026-06-18 16:45:16,462 | _src.PhaseDiagram.new_methodv2 | INFO | Диапазон поиска: unstable=184.5785 bar -> stable=185.5550 bar
2026-06-18 16:45:16,854 | _src.PhaseDiagram.new_methodv2 | WARNING | Давление застряло на итерации 14. P=185.047483
2026-06-18 16:45:16,855 | _src.PhaseDiagram.new_methodv2 | WARNING | Превышен лимит итераций (100). Последнее P=185.0475 bar


ValueError: Length of values (2) does not match length of index (3)

In [ ]:
cce.calculate_parallel()

In [ ]:
cce.calculate_parallel()